# CoT-ENEM — Fase 3 no Colab

Antes de executar: **Ambiente de execução → Alterar tipo de ambiente de execução → GPU**. O notebook usa Qwen2.5-7B-Instruct em 4 bits, grava cada registro no Drive e retoma sem duplicar registros.

In [ ]:
# Edite somente estes valores.
REPOSITORY_URL = "https://github.com/cidadaofred/cot-enem.git"
BRANCH = "master"
LIMIT = None  # None processa toda a base; use 1 primeiro para validar.
DRIVE_ROOT = "/content/drive/MyDrive/cot-enem"


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
for folder in ("data/processed", "outputs/datasets", "outputs/logs"):
    (Path(DRIVE_ROOT) / folder).mkdir(parents=True, exist_ok=True)


In [ ]:
import importlib.util, os, shutil, subprocess
is_colab = importlib.util.find_spec("google.colab") is not None and Path("/content").is_dir()
if not is_colab:
    raise RuntimeError("Este notebook deve ser executado no Google Colab.")
PROJECT = Path("/content/cot-enem").resolve()
if PROJECT != Path("/content/cot-enem"):
    raise RuntimeError(f"Caminho de projeto inseguro: {PROJECT}")
# Saia do checkout antes de removê-lo; o kernel pode continuar nele após uma execução anterior.
os.chdir("/content")
if PROJECT.exists():
    shutil.rmtree(PROJECT)
clone = subprocess.run(
    ["git", "clone", "--depth", "1", "--branch", BRANCH, REPOSITORY_URL, str(PROJECT)],
    text=True,
    capture_output=True,
)
if clone.returncode != 0:
    raise RuntimeError(f"Falha no git clone:\n{clone.stderr.strip()}")
os.chdir(PROJECT)
LOCAL_MODEL_CACHE = "/content/hf-cache"
Path(LOCAL_MODEL_CACHE).mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = LOCAL_MODEL_CACHE
subprocess.run(["python", "-m", "pip", "install", "-q", "-e", ".[colab]"], check=True)


In [ ]:
# Confirma GPU e dependências antes de baixar/carregar o modelo.
subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["python", "-m", "cot_enem.cli", "verify", "--config", "configs/colab.yaml"], check=True)


Envie uma única vez o arquivo local `data/processed/enem_normalized.jsonl` para `Meu Drive/cot-enem/data/processed/`. A célula abaixo interrompe cedo se o arquivo não estiver lá.

In [ ]:
INPUT = Path(DRIVE_ROOT) / "data/processed/enem_normalized.jsonl"
PRIOR_RESULTS = Path(DRIVE_ROOT) / "outputs/datasets/cot_enem_v1.jsonl"
CANDIDATES = Path(DRIVE_ROOT) / "outputs/datasets/phase3_candidates.jsonl"
VOTES = Path(DRIVE_ROOT) / "outputs/datasets/phase3_judge_votes.jsonl"
OUTPUT = Path(DRIVE_ROOT) / "outputs/datasets/cot_enem_specify_ensemble_v1.jsonl"
if not INPUT.exists():
    raise FileNotFoundError(f"Envie enem_normalized.jsonl para: {INPUT}")
print(f"Entrada: {INPUT}")
print(f"Saída persistente: {OUTPUT}")


In [ ]:
command = [
    "python", "-m", "cot_enem.cli", "ensemble-specify",
    "--config", "configs/colab.yaml",
    "--input", str(INPUT),
    "--candidates", str(CANDIDATES),
    "--votes", str(VOTES),
    "--output", str(OUTPUT),
]
if PRIOR_RESULTS.exists():
    print(f"Reutilizando candidatos anteriores de: {PRIOR_RESULTS}")
    command += ["--existing-results", str(PRIOR_RESULTS)]
else:
    print("Sem resultado anterior: os candidatos serão gerados do zero pelo Qwen.")
if LIMIT is not None:
    command += ["--limit", str(LIMIT)]
process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
for line in process.stdout:
    print(line, end="", flush=True)
returncode = process.wait()
if returncode != 0:
    raise RuntimeError(f"ensemble-specify terminou com código {returncode}")


In [ ]:
import json
lines = [line for line in OUTPUT.read_text(encoding="utf-8").splitlines() if line.strip()]
print(f"Registros persistidos: {len(lines)}")
if lines:
    print(json.dumps(json.loads(lines[-1]), ensure_ascii=False, indent=2)[:4000])


## Auditoria da Fase 3

Valida em CPU os quatro JSONL, schemas, duplicidades, referências, três votos por candidato, votação majoritária e completude. Com `LIMIT` definido, a auditoria aceita um processamento parcial consistente.

In [ ]:
import sys
if str(PROJECT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT / "src"))
import yaml
from cot_enem.audit import audit_phase3
with Path("configs/colab.yaml").open("r", encoding="utf-8") as file:
    judge_models = yaml.safe_load(file)["model"]["judge_names"]
summary = audit_phase3(
    INPUT,
    CANDIDATES,
    VOTES,
    OUTPUT,
    judge_models,
    require_complete=LIMIT is None,
)
print("AUDITORIA ESTRUTURAL: APROVADA")
print("=" * 60)
for key, value in summary.items():
    if key == "acceptance_rate":
        print(f"{key:24}: {value:.1f}%")
    else:
        print(f"{key:24}: {value}")
print("=" * 60)
print(f"PROCESSAMENTO COMPLETO: {summary['complete']}")
